# 🛰️ AlphaEarth Foundations → Ollama Explorer
**64-dim satellite embeddings · Source Cooperative COGs · No GEE required**

> *"The AlphaEarth Foundations Satellite Embedding dataset is produced by Google and Google DeepMind."*  
> License: [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/)

**What this notebook does:**
1. Pulls pre-computed 64-dim AEF embeddings from Source Cooperative COGs via HTTP range requests
2. Dequantizes int8 → float32 unit-norm vectors
3. Generates PCA false-color, channel activation, and change maps
4. Sends embedding summaries to Ollama Cloud for LLM interpretation

**No Google Earth Engine. No GCP account. No authentication.**

---
### Quick start
1. **Runtime → Run all** (or `Ctrl+F9`)
2. When prompted, enter your **Ollama Cloud API key** (or skip for extraction-only)
3. Edit the AOI cell to your study area

---
## 1 · Install Dependencies

In [ ]:
#@title Install dependencies (run once per session)
import subprocess, sys

def _run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-500:])
    return result.returncode == 0

print("Installing system GDAL/PROJ...")
_run("apt-get install -qq libgdal-dev gdal-bin libproj-dev 2>/dev/null")

print("Installing Python packages...")
_run("pip install -q rasterio pyproj shapely scikit-learn httpx requests Pillow")

# Verify
import importlib
ok = True
for pkg in ["rasterio", "pyproj", "shapely", "sklearn", "httpx"]:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg}")
    except ImportError:
        print(f"  ✗ {pkg} — FAILED")
        ok = False

if ok:
    print("\n✅ All dependencies ready")
else:
    print("\n⚠️  Some packages failed — try Runtime → Restart and run again")

---
## 2 · Configuration
**Edit this cell** — everything else runs automatically.

In [ ]:
# ── AOI (WGS84 decimal degrees) ──────────────────────────────────────────────
#  Keep the box small (~5-10km wide) for fast COG windowed reads.
#  Larger boxes = more data = slower.

# Preset options — uncomment one or define your own:

# Catonsville, MD
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = -76.755, 39.240, -76.680, 39.290
AOI_NAME = "Catonsville MD"

# Baltimore, MD
# LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = -76.720, 39.250, -76.550, 39.380
# AOI_NAME = "Baltimore MD"

# Washington DC
# LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = -77.120, 38.800, -76.910, 38.990
# AOI_NAME = "Washington DC"

# New York City
# LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = -74.050, 40.680, -73.920, 40.800
# AOI_NAME = "New York City"

# Chesapeake Bay
# LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = -76.500, 38.700, -76.000, 39.100
# AOI_NAME = "Chesapeake Bay"

# ── Years ────────────────────────────────────────────────────────────────────
YEAR_1 = 2023          # Primary year (2017–2024)
YEAR_2 = 2019          # Comparison year for change detection (set None to skip)

# ── Ollama Cloud ─────────────────────────────────────────────────────────────
OLLAMA_MODEL  = "llama3.2"               # Model name on Ollama Cloud
OLLAMA_HOST   = "https://api.ollama.ai"  # Ollama Cloud base URL
# API key — set here or leave blank to be prompted securely below
OLLAMA_API_KEY = ""                      # e.g. "ollama_abc123..."

# ── Task ─────────────────────────────────────────────────────────────────────
# 'interpret' = landscape characterization
# 'change'    = change detection analysis (requires YEAR_2)
# 'report'    = technical methods paragraph
TASK = "change" if YEAR_2 else "interpret"

print(f"AOI:    {AOI_NAME}  [{LON_MIN},{LAT_MIN} → {LON_MAX},{LAT_MAX}]")
print(f"Years:  {YEAR_1}" + (f" → {YEAR_2} (change detection)" if YEAR_2 else ""))
print(f"Model:  {OLLAMA_MODEL} @ {OLLAMA_HOST}")
print(f"Task:   {TASK}")

In [ ]:
#@title API Key (skip if already set above)
import getpass

if not OLLAMA_API_KEY:
    try:
        # Try Colab userdata first (set via Colab secrets panel)
        from google.colab import userdata
        OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY') or ""
        if OLLAMA_API_KEY:
            print("✓ Loaded API key from Colab secrets")
    except Exception:
        pass

if not OLLAMA_API_KEY:
    OLLAMA_API_KEY = getpass.getpass("Ollama Cloud API key (leave blank to skip LLM): ")

if OLLAMA_API_KEY:
    print(f"✓ API key set ({len(OLLAMA_API_KEY)} chars)")
else:
    print("⚠️  No API key — will extract embeddings and visualize, but skip Ollama query")

---
## 3 · Pipeline Core
*Run this cell once — defines all functions used below.*

In [ ]:
import json, logging, os, re, warnings
import numpy as np
import httpx
import requests
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("aef")

import rasterio
from rasterio.windows import from_bounds as window_from_bounds
from pyproj import Transformer
from shapely.geometry import box

# ── Constants ────────────────────────────────────────────────────────────────
SOURCE_COOP_BASE = "https://data.source.coop/tge-labs/aef/v1/annual"
GCS_BASE         = "https://storage.googleapis.com/alphaearth_foundations/satellite_embedding/v1/annual"
NUM_CHANNELS     = 64
NODATA_INT       = -128
_INDEX_CACHE     = None

# ── UTM helpers ──────────────────────────────────────────────────────────────
def wgs84_to_utm_zone(lon, lat):
    return f"{int((lon + 180) / 6) + 1}{'N' if lat >= 0 else 'S'}"

def utm_epsg(zone_str):
    num = int(re.match(r"(\d+)", zone_str).group(1))
    return (32600 if zone_str.endswith("N") else 32700) + num

# ── Tile index ───────────────────────────────────────────────────────────────
def load_index(cache_path="/tmp/aef_index.csv"):
    global _INDEX_CACHE
    if _INDEX_CACHE is not None:
        return _INDEX_CACHE
    if os.path.exists(cache_path):
        log.info("Using cached index")
        raw = open(cache_path).read()
    else:
        url = f"{SOURCE_COOP_BASE}/aef_index.csv"
        log.info("Downloading tile index from %s ...", url)
        r = requests.get(url, timeout=120, stream=True)
        r.raise_for_status()
        raw = r.text
        with open(cache_path, "w") as f:
            f.write(raw)
        log.info("Index cached (%d bytes)", len(raw))
    lines = raw.strip().splitlines()
    header = [h.strip() for h in lines[0].split(",")]
    _INDEX_CACHE = [dict(zip(header, l.split(","))) for l in lines[1:]]
    log.info("Loaded %d tile records", len(_INDEX_CACHE))
    return _INDEX_CACHE

def find_tiles(aoi_box, year, index):
    matched = []
    year_str = str(year)
    for rec in index:
        if year_str not in rec.get("filename", ""):
            continue
        try:
            minx = float(rec.get("minx", rec.get("left", 0)))
            miny = float(rec.get("miny", rec.get("bottom", 0)))
            maxx = float(rec.get("maxx", rec.get("right", 0)))
            maxy = float(rec.get("maxy", rec.get("top", 0)))
        except (ValueError, TypeError):
            continue
        if aoi_box.intersects(box(minx, miny, maxx, maxy)):
            matched.append(rec)
    return matched

def tile_url(filename):
    fname = filename.lstrip("/")
    if fname.startswith("v1/annual/"):
        fname = fname[len("v1/annual/"):]
    return f"{SOURCE_COOP_BASE}/{fname}"

# ── COG read + dequantize ────────────────────────────────────────────────────
def dequantize(raw):
    """int8 [-127..127] → float32 [-1..1]. -128 → NaN."""
    out = raw.astype(np.float32)
    mask = raw == NODATA_INT
    out[mask] = np.nan
    v = ~mask
    out[v] = ((out[v] / 127.5) ** 2) * np.sign(out[v])
    return out

def read_cog(url, lon_min, lat_min, lon_max, lat_max, epsg_code):
    """Windowed read of a single COG tile. Returns (H,W,64) float32 + bool mask."""
    tr = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg_code}", always_xy=True)
    x0, y0 = tr.transform(lon_min, lat_min)
    x1, y1 = tr.transform(lon_max, lat_max)
    left, right  = min(x0,x1), max(x0,x1)
    bottom, top  = min(y0,y1), max(y0,y1)

    vsicurl = f"/vsicurl/{url}"
    with rasterio.open(vsicurl) as src:
        log.info("Opened: %s  shape=%s", url.split("/")[-1], src.shape)
        win = window_from_bounds(left, bottom, right, top, src.transform)
        win = win.intersection(rasterio.windows.Window(0, 0, src.width, src.height))
        raw = src.read(window=win)  # (64, H, W) int8

    data = dequantize(raw)                        # (64, H, W) float32
    mask = ~np.isnan(data).any(axis=0)            # (H, W) bool
    data = np.transpose(data, (1, 2, 0))          # (H, W, 64)
    data[~mask] = 0.0
    log.info("Window: %s  valid=%d/%d", data.shape[:2], mask.sum(), mask.size)
    return data, mask

# ── Analysis ─────────────────────────────────────────────────────────────────
def pca_rgb(emb, mask):
    from sklearn.decomposition import PCA
    H, W, C = emb.shape
    flat = emb.reshape(-1, C)
    valid = flat[mask.reshape(-1)]
    pca = PCA(n_components=3).fit(valid)
    proj = np.zeros((H*W, 3), np.float32)
    proj[mask.reshape(-1)] = pca.transform(valid)
    proj = proj.reshape(H, W, 3)
    rgb = np.zeros_like(proj, np.uint8)
    for i in range(3):
        ch = proj[:,:,i]
        ch_v = ch[mask]
        if ch_v.ptp() > 0:
            ch = (ch - ch_v.min()) / ch_v.ptp()
        rgb[:,:,i] = (ch * 255).clip(0,255).astype(np.uint8)
    return rgb

def cosine_sim_map(emb, ref):
    """Per-pixel cosine similarity to a reference vector."""
    return np.einsum("hwc,c->hw", emb, ref).astype(np.float32)

def cosine_change(emb1, emb2):
    """1 - cosine_sim per pixel. 0=identical, 2=opposite."""
    return (1.0 - np.einsum("hwc,hwc->hw", emb1, emb2)).astype(np.float32)

def summarize(emb, mask, name, year, change_map=None):
    valid = emb[mask]
    ch_mean = valid.mean(axis=0)
    ch_std  = valid.std(axis=0)
    top3    = np.argsort(np.abs(ch_mean))[-3:][::-1].tolist()
    s = {
        "aoi": name, "year": year,
        "valid_px": int(mask.sum()), "total_px": int(mask.size),
        "coverage_pct": round(100 * mask.sum() / mask.size, 1),
        "top3_channels": top3,
        "top3_means": [round(float(ch_mean[i]), 4) for i in top3],
        "mean_variance": round(float(ch_std.mean()), 4),
    }
    if change_map is not None:
        vc = change_map[mask]
        s.update({
            "change_mean":  round(float(vc.mean()), 4),
            "change_std":   round(float(vc.std()), 4),
            "change_p90":   round(float(np.percentile(vc, 90)), 4),
            "pct_high_chg": round(float((vc > 0.1).mean() * 100), 1),
        })
    return s

# ── Ollama Cloud ─────────────────────────────────────────────────────────────
def query_ollama(s1, s2=None, model=OLLAMA_MODEL, host=OLLAMA_HOST,
                 api_key=None, task="interpret"):
    if task == "change" and s2:
        prompt = f"""You are a geospatial analyst interpreting satellite embedding change signals.

Year {s1['year']} embedding summary:
{json.dumps(s1, indent=2)}

Year {s2['year']} embedding summary (same AOI):
{json.dumps(s2, indent=2)}

Provide:
1. What the embedding shift suggests about land-cover or environmental change
2. Which channels show the most change and what that might indicate
3. Confidence level and key caveats
4. Recommended follow-up analyses

Be specific and quantitative. Do not hallucinate land types — reason from the statistics."""

    elif task == "report":
        prompt = f"""You are a remote sensing scientist. Write a concise technical paragraph (3-5 sentences)
suitable for a methods section based on this AEF embedding extraction:
{json.dumps(s1, indent=2)}
Include: AOI, year, embedding dimensionality, coverage, and dominant signal pattern."""

    else:
        prompt = f"""You are a geospatial AI assistant. Interpret these AlphaEarth Foundations
64-dimensional embedding statistics for a study area:
{json.dumps(s1, indent=2)}

1. What do the statistics suggest about this landscape?
2. Which dimensions are most active and why?
3. Data quality notes
4. Suggested downstream analyses"""

    messages = [{"role": "user", "content": prompt}]

    if api_key:
        # Ollama Cloud — OpenAI-compatible endpoint
        url = f"{host.rstrip('/')}/v1/chat/completions"
        headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        payload = {"model": model, "messages": messages, "stream": False}
        try:
            r = httpx.post(url, json=payload, headers=headers, timeout=120)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except httpx.HTTPStatusError as e:
            return f"[HTTP {e.response.status_code}]: {e.response.text[:300]}"
        except Exception as e:
            return f"[Ollama Cloud error]: {e}"
    else:
        # Local Ollama
        url = f"{host.rstrip('/')}/api/chat"
        payload = {"model": model, "messages": messages, "stream": False}
        try:
            r = httpx.post(url, json=payload, timeout=120)
            r.raise_for_status()
            return r.json()["message"]["content"]
        except Exception as e:
            return f"[Ollama local error]: {e}"

print("✅ Pipeline functions loaded")

---
## 4 · Extract Embeddings

In [ ]:
aoi_box = box(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
lon_c   = (LON_MIN + LON_MAX) / 2
lat_c   = (LAT_MIN + LAT_MAX) / 2
utm_zone = wgs84_to_utm_zone(lon_c, lat_c)
epsg     = utm_epsg(utm_zone)
print(f"UTM zone: {utm_zone}  (EPSG:{epsg})")

# Load tile index
print("\nLoading tile index from Source Cooperative...")
index = load_index()
print(f"Index: {len(index)} tile records")

# Year 1
print(f"\nFinding tiles for {YEAR_1}...")
tiles_y1 = find_tiles(aoi_box, YEAR_1, index)
print(f"  → {len(tiles_y1)} tile(s) matched")

if not tiles_y1:
    raise ValueError(f"No tiles found for {AOI_NAME} / {YEAR_1}. Try a larger AOI or different year.")

print(f"Reading COG for {YEAR_1}...")
url_y1 = tile_url(tiles_y1[0]["filename"])
emb_y1, mask_y1 = read_cog(url_y1, LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, epsg)
print(f"  Shape: {emb_y1.shape}  Valid: {mask_y1.sum():,}/{mask_y1.size:,} px ({100*mask_y1.mean():.1f}%)")

# Year 2
emb_y2 = mask_y2 = None
if YEAR_2:
    print(f"\nFinding tiles for {YEAR_2}...")
    tiles_y2 = find_tiles(aoi_box, YEAR_2, index)
    print(f"  → {len(tiles_y2)} tile(s) matched")
    if tiles_y2:
        print(f"Reading COG for {YEAR_2}...")
        url_y2 = tile_url(tiles_y2[0]["filename"])
        emb_y2, mask_y2 = read_cog(url_y2, LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, epsg)
        print(f"  Shape: {emb_y2.shape}  Valid: {mask_y2.sum():,}/{mask_y2.size:,} px")

print("\n✅ Embeddings extracted")

---
## 5 · Visualize

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
plt.rcParams.update({"figure.facecolor": "#0b0f1a", "axes.facecolor": "#131929",
                     "text.color": "#e2e8f0", "axes.labelcolor": "#e2e8f0",
                     "axes.edgecolor": "#1e2d47", "xtick.color": "#64748b",
                     "ytick.color": "#64748b", "grid.color": "#1e2d47"})

ncols = 3 if (emb_y2 is not None) else 2
fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 5.5),
                          facecolor="#0b0f1a", constrained_layout=True)
fig.suptitle(f"AlphaEarth Foundations — {AOI_NAME}",
             color="#e2e8f0", fontsize=13, fontweight="bold", y=1.01)

# PCA false-color Y1
rgb_y1 = pca_rgb(emb_y1, mask_y1)
axes[0].imshow(rgb_y1)
axes[0].set_title(f"PCA False-Color {YEAR_1}\n(PC1→R  PC2→G  PC3→B)",
                   color="#00d4aa", fontsize=10)
axes[0].axis("off")

# Channel activation bar chart
valid_emb = emb_y1[mask_y1]
ch_means  = np.abs(valid_emb.mean(axis=0))   # (64,)
ch_stds   = valid_emb.std(axis=0)

colors = ["#00d4aa" if v > ch_means.mean() + ch_means.std() else
          "#3b82f6" if v > ch_means.mean() else "#1e2d47"
          for v in ch_means]
axes[1].bar(range(64), ch_means, color=colors, alpha=0.9, width=0.8)
axes[1].axhline(ch_means.mean(), color="#f59e0b", lw=1, ls="--", label="mean")
axes[1].set_title(f"Channel Activation |mean| — {YEAR_1}", color="#00d4aa", fontsize=10)
axes[1].set_xlabel("AEF dimension", fontsize=8)
axes[1].set_ylabel("|mean activation|", fontsize=8)
axes[1].legend(fontsize=7)
axes[1].set_xlim(-1, 64)

# Change map
if emb_y2 is not None and mask_y2 is not None:
    change = cosine_change(emb_y1, emb_y2)
    joint  = mask_y1 & mask_y2
    chg_display = np.where(joint, change, np.nan)
    im = axes[2].imshow(chg_display, cmap="RdYlGn_r", vmin=0, vmax=0.3)
    axes[2].set_title(f"Cosine Change {YEAR_1}→{YEAR_2}\n(green=stable  red=changed)",
                       color="#00d4aa", fontsize=10)
    axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], label="1 − cos(t1,t2)", shrink=0.8)

plt.savefig("/tmp/aef_overview.png", dpi=150, bbox_inches="tight",
            facecolor="#0b0f1a")
plt.show()
print("Saved: /tmp/aef_overview.png")

In [ ]:
#@title Cosine similarity search — pick a reference pixel { run: "auto" }
REF_Y = 0.5  #@param {type:"slider", min:0.0, max:1.0, step:0.01}
REF_X = 0.5  #@param {type:"slider", min:0.0, max:1.0, step:0.01}

H, W, _ = emb_y1.shape
ry = int(REF_Y * (H - 1))
rx = int(REF_X * (W - 1))
ref_vec = emb_y1[ry, rx]

sim = cosine_sim_map(emb_y1, ref_vec)
sim_masked = np.where(mask_y1, sim, np.nan)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), facecolor="#0b0f1a")
fig.suptitle(f"Cosine Similarity Search — {AOI_NAME} {YEAR_1}",
             color="#e2e8f0", fontsize=12)

ax[0].imshow(pca_rgb(emb_y1, mask_y1))
ax[0].plot(rx, ry, "w*", ms=14, label="Reference pixel")
ax[0].set_title("PCA (reference = ★)", color="#00d4aa", fontsize=10)
ax[0].axis("off")
ax[0].legend(fontsize=8)

im = ax[1].imshow(sim_masked, cmap="coolwarm", vmin=-1, vmax=1)
ax[1].plot(rx, ry, "w*", ms=14)
ax[1].set_title("Cosine similarity to ★", color="#00d4aa", fontsize=10)
ax[1].axis("off")
plt.colorbar(im, ax=ax[1], label="cosine similarity", shrink=0.8)

plt.tight_layout()
plt.show()

---
## 6 · Summarize + Query Ollama

In [ ]:
# Compute change map if we have two years
change_map = cosine_change(emb_y1, emb_y2) if emb_y2 is not None else None

# Build summaries
s1 = summarize(emb_y1, mask_y1, AOI_NAME, YEAR_1, change_map)
s2 = summarize(emb_y2, mask_y2, AOI_NAME, YEAR_2, change_map) if emb_y2 is not None else None

print("=" * 55)
print(f" Embedding Summary — {AOI_NAME}")
print("=" * 55)
print(json.dumps(s1, indent=2))
if s2:
    print()
    print(json.dumps(s2, indent=2))

# Ollama query
print()
if OLLAMA_API_KEY:
    print(f"Querying {OLLAMA_MODEL} on Ollama Cloud...")
    effective_task = "change" if s2 else TASK
    response = query_ollama(s1, s2,
                            model=OLLAMA_MODEL,
                            host=OLLAMA_HOST,
                            api_key=OLLAMA_API_KEY,
                            task=effective_task)
    print()
    print("=" * 55)
    print(f" 🤖 {OLLAMA_MODEL} — {effective_task}")
    print("=" * 55)
    print(response)
else:
    print("⚠️  Skipping Ollama (no API key). Set OLLAMA_API_KEY in cell 2 to enable.")

---
## 7 · Export

In [ ]:
import json
from PIL import Image
from google.colab import files

# Save PCA PNG
pca_path = f"/tmp/aef_pca_{AOI_NAME.replace(' ','_')}_{YEAR_1}.png"
Image.fromarray(pca_rgb(emb_y1, mask_y1)).save(pca_path)

# Save JSON summary
json_path = f"/tmp/aef_summary_{AOI_NAME.replace(' ','_')}_{YEAR_1}.json"
out = {"summary_y1": s1}
if s2:
    out["summary_y2"] = s2
if OLLAMA_API_KEY and 'response' in dir():
    out["llm_response"] = response
with open(json_path, "w") as f:
    json.dump(out, f, indent=2)

# Save raw embeddings (numpy)
npy_path = f"/tmp/aef_emb_{AOI_NAME.replace(' ','_')}_{YEAR_1}.npy"
np.save(npy_path, emb_y1)

print(f"✅ Saved:")
print(f"   PCA PNG:     {pca_path}")
print(f"   JSON summary:{json_path}")
print(f"   Embeddings:  {npy_path}  shape={emb_y1.shape}")
print()
print("Downloading files...")
files.download(pca_path)
files.download(json_path)
files.download(npy_path)

---
## 8 · Extras
Optional cells for deeper analysis.

In [ ]:
#@title UMAP projection of embedding space (optional — installs umap-learn)
!pip install -q umap-learn
import umap

MAX_SAMPLES = 5000  # subsample for speed
valid_flat = emb_y1[mask_y1]  # (N, 64)
if len(valid_flat) > MAX_SAMPLES:
    idx = np.random.choice(len(valid_flat), MAX_SAMPLES, replace=False)
    sample = valid_flat[idx]
else:
    sample = valid_flat

print(f"Running UMAP on {len(sample)} pixels...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
embedding_2d = reducer.fit_transform(sample)

fig, ax = plt.subplots(figsize=(8, 6), facecolor="#0b0f1a")
sc = ax.scatter(embedding_2d[:,0], embedding_2d[:,1],
                c=np.arange(len(sample)), cmap="plasma", s=2, alpha=0.6)
ax.set_title(f"UMAP — {AOI_NAME} {YEAR_1} (n={len(sample)})",
             color="#00d4aa", fontsize=11)
ax.set_xlabel("UMAP-1", fontsize=9)
ax.set_ylabel("UMAP-2", fontsize=9)
plt.colorbar(sc, ax=ax, label="pixel index", shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
#@title K-Means clustering of embedding space
N_CLUSTERS = 6  #@param {type:"integer"}

from sklearn.cluster import MiniBatchKMeans

valid_flat = emb_y1[mask_y1]
print(f"Clustering {len(valid_flat):,} valid pixels into {N_CLUSTERS} classes...")
km = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=3)
labels_valid = km.fit_predict(valid_flat)

H, W, _ = emb_y1.shape
label_map = np.full((H, W), -1, dtype=np.int16)
label_map[mask_y1] = labels_valid

fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor="#0b0f1a")
fig.suptitle(f"K-Means Clustering (k={N_CLUSTERS}) — {AOI_NAME} {YEAR_1}",
             color="#e2e8f0", fontsize=12)

axes[0].imshow(pca_rgb(emb_y1, mask_y1))
axes[0].set_title("PCA False-Color", color="#00d4aa", fontsize=10)
axes[0].axis("off")

cmap_clusters = plt.cm.get_cmap("tab10", N_CLUSTERS)
cluster_display = np.where(mask_y1, label_map, np.nan).astype(float)
im = axes[1].imshow(cluster_display, cmap=cmap_clusters, vmin=0, vmax=N_CLUSTERS-1)
axes[1].set_title(f"Cluster Map", color="#00d4aa", fontsize=10)
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], ticks=range(N_CLUSTERS), label="cluster", shrink=0.8)

plt.tight_layout()
plt.show()

# Cluster stats
print("\nCluster sizes:")
for k in range(N_CLUSTERS):
    n = (labels_valid == k).sum()
    print(f"  Cluster {k}: {n:,} px ({100*n/len(labels_valid):.1f}%)")